In [ ]:
# IPython magic commands
%load_ext autoreload
%autoreload 2

# Standard library imports
import os
from pathlib import Path

# Third-party imports
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

# AEON imports
from aeon.schema.schemas import social02
from swc.aeon.io import api as aeon_api

# Custom utilities imports
from data_io_utils import load_data_from_parquet, save_all_experiment_data

# Definitions

In [ ]:
data_dir = Path("/ceph/aeon/aeon/code/scratchpad/methods_paper_data")
save_dir = Path("/ceph/aeon/aeon/code/scratchpad/anaya/rl_modelling_results")
os.makedirs(data_dir, exist_ok=True)
os.makedirs(save_dir, exist_ok=True)
cm2px = 5.2  # 1 cm = 5.2 px roughly in aeon arenas
light_off, light_on = 7, 20  # 7am to 7pm
fps = 50

In [ ]:
experiments = [
    {"name": "social0.2-aeon3", "presocial_start": '2024-01-31 11:00:00', "presocial_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 16:00:00', "social_end": '2024-02-23 13:00:00', "postsocial_start": '2024-02-25 17:00:00', "postsocial_end": '2024-03-02 14:00:00'},
    {"name": "social0.2-aeon4", "presocial_start": '2024-01-31 11:00:00', "presocial_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 17:00:00', "social_end": '2024-02-23 12:00:00', "postsocial_start": '2024-02-25 18:00:00', "postsocial_end": '2024-03-02 13:00:00'},
    {"name": "social0.3-aeon3", "presocial_start": '2024-06-08 19:00:00', "presocial_end": '2024-06-17 13:00:00', "social_start": '2024-06-25 11:00:00', "social_end": '2024-07-06 13:00:00', "postsocial_start": '2024-07-07 16:00:00', "postsocial_end": '2024-07-14 14:00:00'},
    {"name": "social0.3-aeon4", "presocial_start": '2024-06-08 19:00:00', "presocial_end": '2024-06-17 14:00:00', "social_start": '2024-06-19 12:00:00', "social_end": '2024-07-03 14:00:00', "postsocial_start": '2024-07-04 11:00:00', "postsocial_end": '2024-07-13 12:00:00'},
    {"name": "social0.4-aeon3", "presocial_start": '2024-08-16 17:00:00', "presocial_end": '2024-08-24 10:00:00', "social_start": '2024-08-28 11:00:00', "social_end": '2024-09-09 13:00:00', "postsocial_start": '2024-09-09 18:00:00', "postsocial_end": '2024-09-22 16:00:00'},
    {"name": "social0.4-aeon4", "presocial_start": '2024-08-16 15:00:00', "presocial_end": '2024-08-24 10:00:00', "social_start": '2024-08-28 10:00:00', "social_end": '2024-09-09 01:00:00', "postsocial_start": '2024-09-09 15:00:00', "postsocial_end": '2024-09-22 16:00:00'}
]
experiment = experiments[0]

In [ ]:
def load_experiment_data(
    data_dir: str,
    experiment: dict | None = None,
    periods: list | None = None,
    data_types: list[str] = ['rfid', 'position'],
    trim_days: int | None = None
) -> dict:
    """
    Load all data types for specified periods of an experiment.
    
    Parameters:
    - experiment: experiment dict with period start/end times
    - periods: list of periods to load
    - data_types: list of data types to load
    - data_dir: directory containing data files
    - trim_days: Optional number of days to trim from start (None = no trim)
    
    Returns:
    - Dictionary containing dataframes for each period/data type combination
    """
    
    result = {}

    if periods is None:
        periods = [None]
    
    for period in periods:
        for data_type in data_types:
            print(f"Loading {period} {data_type} data...")
            
            # Load data
            if experiment is not None:
                experiment_name = experiment["name"]
            else:
                experiment_name = None
            df = load_data_from_parquet(
                experiment_name=experiment_name,
                period=period,
                data_type=data_type,
                data_dir=data_dir,
                set_time_index=(data_type == 'position')
            )
            
            # Trim if requested
            if trim_days is not None and len(df) > 0:
                if data_type == 'rfid':
                    start_time = df['chunk_start'].min()
                    end_time = start_time + pd.Timedelta(days=trim_days)
                    df = df[df['chunk_start'] < end_time]
                if data_type == 'foraging':
                    start_time = df['start'].min()
                    end_time = start_time + pd.Timedelta(days=trim_days)
                    df = df[df['start'] < end_time]
                if data_type == 'position':
                    start_time = df.index.min()
                    end_time = start_time + pd.Timedelta(days=trim_days)
                    df = df.loc[df.index < end_time]
                
                print(f"  Trimmed to {trim_days} days: {len(df)} records")
            
            # Store in result
            key = f"{period}_{data_type}"
            result[key] = df
            
            # For position data, handle duplicates
            if data_type == 'position' and len(df) > 0:
                original_len = len(df)
                df = df.reset_index()
                df = df.drop_duplicates(subset=['time', 'identity_name'])
                df = df.set_index('time')
                result[key] = df
                if len(df) < original_len:
                    print(f"  Removed duplicates: {original_len} -> {len(df)}")
    
    return result

# Load and prepare data

In [ ]:
data = load_experiment_data(
    experiment=experiment,
    data_dir=data_dir,
    periods=['social'],
    data_types=["patchinfo", "positiondenoised"]
)

social_patchinfo_df = data['social_patchinfo']
social_position_df = data['social_positiondenoised']
social_position_df.set_index('time', inplace=True)
social_position_df.sort_index(inplace=True)

In [ ]:
"""Find unique patch configurations and their counts"""

# Group by block_start and aggreGate patch_name and patch_rate combinations
config_df = social_patchinfo_df.groupby('block_start').apply(
    lambda x: tuple(sorted(zip(x['patch_name'], x['patch_rate']), key=lambda item: item[0])),
    include_groups=False
).reset_index(name='config')

# Count occurrences and sort by patch rates
config_counts = config_df['config'].value_counts().sort_index(
    key=lambda idx: idx.map(lambda x: (x[0][1], x[1][1], x[2][1]))
)
n_configs = len(config_counts)

# Create numbered mapping and add to dataframe
config_mapping = {config: i+1 for i, config in enumerate(config_counts.index)}
config_df['config_num'] = config_df['config'].map(config_mapping)

# Display results
print(f"Number of unique configs: {n_configs}\n")
print("Config#  Count  Patch1   Patch2   Patch3")
print("-" * 45)
for config, count in config_counts.items():
    rates = [f"{rate:.4f}" for _, rate in config]
    print(f"  {config_mapping[config]:2d}     {count:2d}    {rates[0]}   {rates[1]}   {rates[2]}")

display(config_df)

In [ ]:
"""Add block_start and config_num to social_position_df"""

# Merge config_num and block_start based on block_start times
merge_result = pd.merge_asof(
    social_position_df.reset_index()[['time']],
    config_df[['block_start', 'config_num']].sort_values('block_start'),
    left_on='time',
    right_on='block_start',
    direction='backward'
)

social_position_df['config_num']   = merge_result['config_num'].values
social_position_df['block_start']  = merge_result['block_start'].values

display(social_position_df)

In [ ]:
"""Make state df"""

# Choose focal mouse (0 or 1)
focal_identity = social_position_df['identity_name'].unique()[0]  # or [1]

# Split into self and other, keeping only what we need (include time + context)
self_df = social_position_df[social_position_df['identity_name'] == focal_identity]\
    .reset_index()[['time', 'experiment_name', 'block_start', 'config_num', 'x', 'y']]\
    .rename(columns={'x': 'x_self', 'y': 'y_self'})

other_df = social_position_df[social_position_df['identity_name'] != focal_identity]\
    .reset_index()[['time', 'experiment_name', 'block_start', 'x', 'y']]\
    .rename(columns={'x': 'x_other', 'y': 'y_other'})

# Exact merge on time + experiment + block (no asof, no tolerance)
state_df = self_df.merge(
    other_df,
    on=['time', 'experiment_name', 'block_start'],
    how='inner'
)

# Add velocities (finite differences within each experiment/block)
dt = 1.0 / fps
state_df[['vx_self', 'vy_self']] = state_df.groupby(
    ['experiment_name', 'block_start']
)[['x_self', 'y_self']].diff() / dt
state_df[['vx_other', 'vy_other']] = state_df.groupby(
    ['experiment_name', 'block_start']
)[['x_other', 'y_other']].diff() / dt

# Relative position and velocity (other - self)
state_df['dx']  = state_df['x_other'] - state_df['x_self']
state_df['dy']  = state_df['y_other'] - state_df['y_self']
state_df['dvx'] = state_df['vx_other'] - state_df['vx_self']
state_df['dvy'] = state_df['vy_other'] - state_df['vy_self']

# Drop the first frame of each block where velocities are NaN
state_df = state_df.dropna(subset=['vx_self', 'vy_self', 'vx_other', 'vy_other'])

display(state_df)

In [ ]:
"""Build transitions (s, a, s_next, done, c) from state_df"""

k = 10  # Δt = 200 ms at 50 Hz

state_cols = ["x_self", "y_self", "vx_self", "vy_self", "dx", "dy", "dvx", "dvy"]

# Index within each block and block length
state_df["block_idx"] = state_df.groupby(["experiment_name", "block_start"]).cumcount()
state_df["block_len"] = state_df.groupby(["experiment_name", "block_start"])["block_idx"].transform("max") + 1

# Next-state columns via k-step shift within each block
group_keys = ["experiment_name", "block_start"]
for col in state_cols:
    state_df[f"{col}_next"] = state_df.groupby(group_keys)[col].shift(-k)

# Keep only rows where a next state exists
mask = state_df["x_self_next"].notna()
trans_df = state_df[mask]

# s_t and s_{t+k}
s = trans_df[state_cols].to_numpy(dtype="float32")
s_next = trans_df[[f"{c}_next" for c in state_cols]].to_numpy(dtype="float32")

# k-step action: displacement of self
a = np.stack([
    trans_df["x_self_next"].values - trans_df["x_self"].values,
    trans_df["y_self_next"].values - trans_df["y_self"].values,
], axis=1).astype("float32")

# done: last valid transition in each block (after which no further k-step exists)
done = (trans_df["block_idx"] + k == trans_df["block_len"] - 1).to_numpy()

# Environment config (config_num at time t)
c = trans_df["config_num"].to_numpy(dtype="int64")
n_configs = c.max()
c_onehot = np.eye(n_configs)[c - 1].astype("float32")

print("Transitions:", len(s), "  States dim:", s.shape[1], "  Action dim:", a.shape[1])

In [ ]:
"""Prepare data for imitation policy training"""

# Training parameters - adjust as needed
batch_size = 8192
epochs = 2
additional_epochs = 0 # if >0 and checkpoint exists, number of additional epochs to train
log_every = 500 # number of batches between logging

# Define device and check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU name:", torch.cuda.get_device_name(0))

# Build phi, a low-dimensional subset of state features most relevant for action selection
phi_cols = ["x_self", "y_self", "dx", "dy"]
phi_np = trans_df[phi_cols].to_numpy(dtype="float32")

# Convert arrays to torch
phi_tensor = torch.from_numpy(phi_np).float()
a_tensor = torch.from_numpy(a).float()

# z-score normalisation
phi_mean, phi_std = phi_tensor.mean(0), phi_tensor.std(0) + 1e-6
a_mean, a_std = a_tensor.mean(0), a_tensor.std(0) + 1e-6

phi_norm = (phi_tensor - phi_mean) / phi_std
actions_norm = (a_tensor - a_mean) / a_std

# Add env config one-hot encoding to phi
c_onehot_tensor = torch.from_numpy(c_onehot).float()
phi_full = torch.cat([phi_norm, c_onehot_tensor], dim=1)

# Pack into a dataset
imitation_dataset = torch.utils.data.TensorDataset(
    phi_full, # [N, 4 + n_configs]
    actions_norm,  # [N, 2]
)

# Batching and shuffling
imitation_loader = torch.utils.data.DataLoader(
    imitation_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=(device.type == "cuda")
)

In [ ]:
"""Define imitation policy network and optimiser"""

class ImitationPolicy(nn.Module):
    def __init__(self, input_dim=4, # φ(s) = (x_self, y_self, dx, dy)
                 hidden_dim=128, 
                 action_dim=2 # action = (Δx, Δy)
                 ):
        super().__init__()
        # 2-layer MLP: φ(s) → 128-dim hidden features
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        ) 
        # Final linear layer mapping hidden features → mean of Gaussian over actions
        self.mu_head = nn.Linear(hidden_dim, action_dim) 
        # Learnable log-standard-deviation for each action dimension (σ_x, σ_y)
        # Using a global std (state-independent) for stability
        # We store log σ (unconstrained real values) and exponentiate it in forward() (to ensure positivity)
        self.log_std = nn.Parameter(torch.zeros(action_dim)) 

    def forward(self, phi):
        # Compute hidden features h = MLP(φ)
        h = self.net(phi)
        # Predict the Gaussian mean μ(φ)
        mu = self.mu_head(h)
        # Clamp log σ to keep the Gaussian variance in a reasonable range.
        # This prevents the model from inflating σ (collapse to a huge, flat Gaussian).
        # exp(log_std) ensures σ > 0, since log σ is unconstrained.
        # log_std = torch.clamp(self.log_std, -3.0, 0.0) # σ in [0.05, 1.0]
        # std = torch.exp(log_std)
        std = torch.exp(self.log_std) # clamp removed but can be re-added if needed
        return mu, std

    def log_prob(self, phi, a):
        # Compute μ and σ for the given φ(s) (forward pass)
        mu, std = self(phi)
        # Define π(a | φ(s)) as an independent Normal over Δx and Δy
        dist = torch.distributions.Normal(mu, std) 
        # Log-prob per dimension → sum = joint log-prob
        # p(a∣s)=p(Δx∣s)p(Δy∣s) so logp(a∣s)=logp(Δx∣s)+logp(Δy∣s)
        return dist.log_prob(a).sum(-1)

# Instantiate imitation policy and optimiser
policy = ImitationPolicy(input_dim=phi_full.shape[1], action_dim=actions_norm.shape[1]).to(device)
optimiser = optim.Adam(policy.parameters(), lr=1e-3)

In [ ]:
"""Fit imitation policy"""

# Check if saved file exists
load = False
save_path = save_dir / "imitation_policy_w_c_2.pt"
if save_path.exists():
    load = True

if load:
    print("Loading imitation policy from disk...")
    checkpoint = torch.load(save_path, map_location=device)
    policy.load_state_dict(checkpoint["policy_net"])
    optimiser.load_state_dict(checkpoint["policy_opt"])
    
    phi_mean = checkpoint["phi_mean"]
    phi_std = checkpoint["phi_std"]
    a_mean = checkpoint["a_mean"]
    a_std = checkpoint["a_std"]

    start_epoch = checkpoint.get("epoch", 0)

    if additional_epochs <= 0 :
        print(f"Model loaded (epoch {start_epoch}). No additional training requested, skipping training.")
        do_train = False
    else:
        total_epochs = start_epoch + additional_epochs
        print(
            f"Model loaded (epoch {start_epoch}). "
            f"Continuing training for {additional_epochs} more epochs "
            f"(up to epoch {total_epochs})."
        )
        do_train = True
else:
    print("No existing checkpoint, training from scratch...")
    start_epoch = 0
    total_epochs = epochs
    do_train = True
    
if do_train:
    print("Training imitation policy...")

    # Train π(a | φ(s)) via maximum likelihood estimation (MLE)
    policy.train()
    for epoch in range(start_epoch, total_epochs):
        running_loss = 0.0
        n_batches = 0
        for step, (phi_batch, a_batch) in enumerate(tqdm(imitation_loader, desc=f"Epoch {epoch+1}", miniters=100)):
            # Move to GPU (or CPU)
            phi_batch = phi_batch.to(device, non_blocking=True) # [batch_size, 4]
            a_batch   = a_batch.to(device, non_blocking=True) # [batch_size, 2]

            # Compute log-probabilities and loss
            logp = policy.log_prob(phi_batch, a_batch) # [B]
            loss = -logp.mean() # maximise log-likelihood

            # Gradient step
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()

            # Track loss to print progress
            running_loss += loss.item()
            n_batches += 1

            if (step + 1) % log_every == 0:
                print(
                    f"Epoch {epoch+1}, Step {step+1}: "
                    f"loss={running_loss / n_batches:.4f}"
                )

        print(
            f"\n\nEpoch {epoch+1} done."
            f"loss={running_loss / n_batches:.4f}"
        )
        print("---")

        # Save checkpoint after each epoch
        checkpoint = {
            "policy_net": policy.state_dict(),
            "policy_opt": optimiser.state_dict(),
            "phi_cols": phi_cols,
            "phi_mean": phi_mean,
            "phi_std": phi_std,
            "a_mean": a_mean,
            "a_std": a_std,
            "epoch": epoch + 1,
        }

        # Always keep a "latest" checkpoint
        torch.save(checkpoint, save_path)

        # Also save a per-epoch checkpoint
        epoch_path = save_path.with_name(save_path.stem + f"_epoch{epoch+1}.pt")
        torch.save(checkpoint, epoch_path)
        print(f"Saved epoch {epoch+1} checkpoint to {epoch_path}")
    
    print(f"Saved final imitation policy to {save_path}")

In [ ]:
"""Compute rewards r(s,a) = log π(a | φ(s)) for all transitions"""

# Put network in inference mode (disables training-specific behaviour like dropout/batchnorm)
policy.eval()

# Process in batches to avoid OOM
batch_size = 8192
n_samples = phi_full.shape[0]
logp_list = []

# Disable gradient tracking (no graph → faster + lower memory)
with torch.no_grad():
    for i in tqdm(range(0, n_samples, batch_size), desc="Computing log probs"):
        # Get batch slice
        end_idx = min(i + batch_size, n_samples)
        phi_tensor = phi_full[i:end_idx].to(device) # [batch_size, 4 + n_configs]
        a_tensor = actions_norm[i:end_idx].to(device) # [batch_size, 2]
        # Compute log probs for this batch
        logp_batch = policy.log_prob(phi_tensor, a_tensor) # [batch_size] log π(a|φ(s))
        # Move to CPU immediately to free GPU memory
        logp_list.append(logp_batch.cpu().numpy())

# Concatenate all batches
r = np.concatenate(logp_list).astype("float32")

# Sanity check
print("\nReward stats for sanity check:")
print(f"max: {r.max():.3f} (expected ≈ -0.5 to -2)")
print(f"mean: {r.mean():.3f} (expected ≈ -2 to -4)")
print(f"std: {r.std():.3f} (expected ≈ 5–15)")
print(f"min: {r.min():.1f} (very negative outliers are normal)")

print("\nNumerical issues:")
print(f"NaN: {np.isnan(r).sum()} (should be 0)")
print(f"inf: {np.isinf(r).sum()} (should be 0)")

log_std=policy.log_std.detach().cpu().numpy()
std=np.exp(log_std)
print("\nPolicy std:")
print(f"log_std: {log_std} (expected in [-3,0])")
print(f"std: {std} (≈1 expected since actions were normalised)")

In [ ]:
"""Load metadata and define arena boundaries"""

exp, acquisition_computer = experiment["name"].split('-', 1)
acquisition_computer = acquisition_computer.upper()
root_path = f"/ceph/aeon/aeon/data/raw/{acquisition_computer}/{exp}"
metadata_reader = social02.Metadata
metadata = aeon_api.load(root_path, metadata_reader)['metadata'].iloc[0]
inner_radius = float(metadata.ActiveRegion.ArenaInnerRadius)
outer_radius = float(metadata.ActiveRegion.ArenaOuterRadius)
center_x = float(metadata.ActiveRegion.ArenaCenter.X)
center_y = float(metadata.ActiveRegion.ArenaCenter.Y)
nest_corner_1 = metadata.ActiveRegion.NestRegion.ArrayOfPoint[0]
nest_corner_2 = metadata.ActiveRegion.NestRegion.ArrayOfPoint[1]
nest_corner_3 = metadata.ActiveRegion.NestRegion.ArrayOfPoint[2]
nest_corner_4 = metadata.ActiveRegion.NestRegion.ArrayOfPoint[3]
patch1_corner_1 = metadata.ActiveRegion.Patch1Region.ArrayOfPoint[0]
patch1_corner_2 = metadata.ActiveRegion.Patch1Region.ArrayOfPoint[1]
patch1_corner_3 = metadata.ActiveRegion.Patch1Region.ArrayOfPoint[2]
patch1_corner_4 = metadata.ActiveRegion.Patch1Region.ArrayOfPoint[3]
patch2_corner_1 = metadata.ActiveRegion.Patch2Region.ArrayOfPoint[0]
patch2_corner_2 = metadata.ActiveRegion.Patch2Region.ArrayOfPoint[1]
patch2_corner_3 = metadata.ActiveRegion.Patch2Region.ArrayOfPoint[2]
patch2_corner_4 = metadata.ActiveRegion.Patch2Region.ArrayOfPoint[3]
patch3_corner_1 = metadata.ActiveRegion.Patch3Region.ArrayOfPoint[0]
patch3_corner_2 = metadata.ActiveRegion.Patch3Region.ArrayOfPoint[1]
patch3_corner_3 = metadata.ActiveRegion.Patch3Region.ArrayOfPoint[2]
patch3_corner_4 = metadata.ActiveRegion.Patch3Region.ArrayOfPoint[3]

def get_validity_mask(xx, yy, metadata):
    # Extract geometry
    inner_radius = float(metadata.ActiveRegion.ArenaInnerRadius)
    outer_radius = float(metadata.ActiveRegion.ArenaOuterRadius)
    center_x = float(metadata.ActiveRegion.ArenaCenter.X)
    center_y = float(metadata.ActiveRegion.ArenaCenter.Y)
    
    # Nest geometry
    nest_pts = metadata.ActiveRegion.NestRegion.ArrayOfPoint
    nest_xs = [float(p.X) for p in nest_pts]
    nest_ys = [float(p.Y) for p in nest_pts]
    
    # Calculate distances
    dx = xx - center_x
    dy = yy - center_y
    r = np.sqrt(dx**2 + dy**2)
    
    # Define zones
    mask_inner = (r <= inner_radius)
    mask_corridor = (r >= inner_radius) & (r <= outer_radius)
    mask_nest = (
        (xx >= min(nest_xs)) & (xx <= max(nest_xs)) & 
        (yy >= min(nest_ys)) & (yy <= max(nest_ys))
    )
    
    return mask_inner | mask_corridor | mask_nest

In [ ]:
"""Plot Imitation Policy for different fixed other positions"""

# Define locations
nest_x = (float(nest_corner_1.X) + float(nest_corner_3.X)) / 2
nest_y = (float(nest_corner_1.Y) + float(nest_corner_3.Y)) / 2
patch1_x = (float(patch1_corner_1.X) + float(patch1_corner_3.X)) / 2
patch1_y = (float(patch1_corner_1.Y) + float(patch1_corner_3.Y)) / 2
patch2_x = (float(patch2_corner_1.X) + float(patch2_corner_3.X)) / 2
patch2_y = (float(patch2_corner_1.Y) + float(patch2_corner_3.Y)) / 2
patch3_x = (float(patch3_corner_1.X) + float(patch3_corner_3.X)) / 2
patch3_y = (float(patch3_corner_1.Y) + float(patch3_corner_3.Y)) / 2
corridor_r = (inner_radius + outer_radius) / 2

locations = {
    "Arena Center": (center_x, center_y),
    "Nest Area": (nest_x, nest_y),
    "Gate": (center_x - corridor_r, center_y),
    "Patch 1": (patch1_x, patch1_y),
    "Patch 2": (patch2_x, patch2_y),
    "Patch 3": (patch3_x, patch3_y),
}
n_locs = len(locations)

# Load image and determine global bounds
arena_img = plt.imread("arena.png")
img_h, img_w = arena_img.shape[:2]
extent_img = [0, img_w, 0, img_h]

# Setup grid
nx, ny = 30, 30
xs = np.linspace(0, img_w, nx)
ys = np.linspace(0, img_h, ny)
xx, yy = np.meshgrid(xs, ys)

# Prepare figure
fig, axes = plt.subplots(
    n_configs, n_locs,
    figsize=(4 * n_locs, 4 * n_configs),
    squeeze=False,
    dpi=300 
)

# Loop over configs (rows) and locations (columns)
unique_configs = list(config_counts.index)
for cfg_idx, cfg_tuple in enumerate(unique_configs):
    rates = [f"{rate:.4f}" for (_, rate) in cfg_tuple]
    # Format: (Rate1, Rate2, Rate3)
    cfg_name = f"({rates[0]}, {rates[1]}, {rates[2]})"

    # One-hot encoding for config
    c_grid = np.zeros((nx * ny, n_configs), dtype="float32")
    c_grid[:, cfg_idx] = 1.0
    c_grid_tensor = torch.from_numpy(c_grid).to(device)

    for loc_j, (loc_name, (other_x, other_y)) in enumerate(locations.items()):
        ax = axes[cfg_idx, loc_j]
        
        print(f"Computing imitation policy for config {cfg_name} with fixed other at {loc_name}...")

        # Construct Input (phi = [x, y, dx, dy])
        phi_grid = np.zeros((nx * ny, 4), dtype="float32")
        phi_grid[:, 0] = xx.ravel() # x_self
        phi_grid[:, 1] = yy.ravel() # y_self
        phi_grid[:, 2] = other_x - xx.ravel() # dx
        phi_grid[:, 3] = other_y - yy.ravel() # dy
        
        # Normalize (using phi stats)
        phi_tensor = torch.from_numpy(phi_grid).to(device)
        phi_norm = (phi_tensor - phi_mean.to(device)) / phi_std.to(device)

        # Add env config one-hot encoding
        phi_full = torch.cat([phi_norm, c_grid_tensor], dim=1)

        # Query Imitation Policy
        policy.eval()
        with torch.no_grad():
            mu, _ = policy(phi_full)
            mu = mu.cpu().numpy()

        # Un-normalize actions
        a_std_np = a_std.detach().cpu().numpy()
        a_mean_np = a_mean.detach().cpu().numpy()
        actions = (mu * a_std_np) + a_mean_np

        u = actions[:, 0].reshape(ny, nx)
        v = actions[:, 1].reshape(ny, nx)

        # Apply masking
        valid_start = get_validity_mask(xx, yy, metadata)
        u = np.where(valid_start, u, np.nan)
        v = np.where(valid_start, v, np.nan)

        # Plot background
        if arena_img is not None:
            ax.imshow(arena_img, origin="lower", alpha=0.5, extent=extent_img)

        # Plot self movement
        ax.quiver(xx, yy, u, v, color='blue', scale=None, scale_units='inches')

        # Plot fixed other mouse
        ax.plot(other_x, other_y, 'b*', markersize=25, markeredgecolor='white', label=f"Other ({loc_name})")

        # Labeling Logic:
        # Column Labels (Top Row only)
        if cfg_idx == 0:
            ax.set_title(loc_name, fontsize=14, fontweight='bold')
        else:
            ax.set_title("") # Clear title for other rows

        # Row Labels (Left Column only)
        if loc_j == 0:
            # Rotated 90 degrees, adjusted labelpad to 20 since it's vertical now
            ax.set_ylabel(f"Config\n{cfg_name}", fontsize=12, fontweight='bold', rotation=90, labelpad=20)
            # Remove standard y ticks to clean up
            ax.set_yticks([]) 
        else:
            ax.set_ylabel("")
            ax.set_yticks([])

        # Remove x ticks for all plots to clean up
        ax.set_xticks([])
        
        # Lock view to full arena dimensions
        ax.set_xlim(0, img_w)
        ax.set_ylim(0, img_h)
        
        ax.set_aspect('equal')

# Add a single legend for the whole figure
# We create a dummy handle to represent the "Other" mouse star
legend_elements = [Line2D([0], [0], marker='*', color='w', label='Other Mouse',
                          markerfacecolor='b', markersize=15)]
fig.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(0.0, 1.0))

plt.tight_layout()
plt.show()

In [ ]:
"""Prepare data for IQL"""

# Training parameters - adjust as needed
batch_size = 4096
epochs = 6
additional_epochs = 0 # if >0 and checkpoint exists, number of additional epochs to train
log_every = 500 # number of batches between logging

# Define device and check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU name:", torch.cuda.get_device_name(0))

# Convert arrays to torch
s_tensor = torch.from_numpy(s).float()
a_tensor = torch.from_numpy(a).float()
r_tensor = torch.from_numpy(r).float()
s_next_tensor = torch.from_numpy(s_next).float()
done_tensor = torch.from_numpy(done.astype("float32"))

# z-score normalisation
s_mean, s_std = s_tensor.mean(0), s_tensor.std(0) + 1e-6
a_mean, a_std = a_tensor.mean(0), a_tensor.std(0) + 1e-6

s_norm = (s_tensor - s_mean) / s_std
s_next_norm = (s_next_tensor - s_mean) / s_std
a_norm = (a_tensor - a_mean) / a_std

# Add env config one-hot encoding to s
c_onehot_tensor = torch.from_numpy(c_onehot).float()
s_full = torch.cat([s_norm, c_onehot_tensor], dim=1)
s_next_full = torch.cat([s_next_norm, c_onehot_tensor], dim=1)

# Pack into a dataset
iql_dataset = torch.utils.data.TensorDataset(
    s_full, # states [N, 8 + n_configs]
    a_norm, # actions [N, 2]
    r_tensor.unsqueeze(-1), # rewards [N, 1]
    s_next_full, # next states [N, 8 + n_configs]
    done_tensor.unsqueeze(-1) # done flags [N, 1]
)

# Batching and shuffling
iql_loader = torch.utils.data.DataLoader(
    iql_dataset, 
    batch_size=batch_size, 
    shuffle=True,
    pin_memory=(device.type == "cuda")
)

In [ ]:
"""Define IQL networks and optimisers"""

# Network parameters - adjust as needed
state_dim = s_full.shape[1]
action_dim = a_norm.shape[1]
hidden_dim = 256
gamma = 0.99 # discount factor
tau_expectile = 0.7 # expectile parameter for value regression
beta = 3.0 # temperature for advantage weights
target_update_rate = 0.005 # Polyak update rate for target nets

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        # 3-layer MLP
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.net(x)

# Value network V(s)
class ValueNet(nn.Module):
    def __init__(self, state_dim, hidden_dim):
        super().__init__()
        self.body = MLP(state_dim, hidden_dim, 1)

    def forward(self, s):
        return self.body(s).squeeze(-1) # [batch_size]

# Q-value network Q(s,a)
class QNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim):
        super().__init__()
        self.body = MLP(state_dim + action_dim, hidden_dim, 1)

    def forward(self, s, a):
        sa = torch.cat([s, a], dim=-1)
        return self.body(sa).squeeze(-1) # [batch_size]

# Gaussian policy network π(a | s)
class GaussianPolicy(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.mu_head = nn.Linear(hidden_dim, action_dim)
        self.log_std = nn.Parameter(torch.zeros(action_dim)) # global log σ

    def forward(self, s):
        h = self.net(s)
        mu = self.mu_head(h)
        std = torch.exp(self.log_std)
        return mu, std

    def log_prob(self, s, a):
        mu, std = self(s)
        dist = torch.distributions.Normal(mu, std)
        return dist.log_prob(a).sum(-1)  # [batch_size]

# Polyak averaging utility
# Called once per batch to keep the target net slowly tracking the online net
# Implements: θ_target ← (1 - τ) * θ_target + τ * θ_source
def polyak_update(source, target, tau):
    with torch.no_grad():
        # Update target network one parameter at a time
        for p, p_targ in zip(source.parameters(), target.parameters()): 
            p_targ.data.mul_(1 - tau).add_(tau * p.data)

# Instantiate networks
value_net = ValueNet(state_dim, hidden_dim).to(device)
# A slow-moving copy of value_net (the “target value network”)
# This gives a stable bootstrap term r + γ * V_target(s′) during Q-learning and prevents divergence.
value_target = ValueNet(state_dim, hidden_dim).to(device)
value_target.load_state_dict(value_net.state_dict())

# Two separate Q networks (the “double-Q” trick)
# Double-Q reduces overestimation bias by taking the conservative target min(Q1(s,a), Q2(s,a))
# This is standard in modern actor–critic RL
q1_net = QNet(state_dim, action_dim, hidden_dim).to(device)
q2_net = QNet(state_dim, action_dim, hidden_dim).to(device)

policy_net = GaussianPolicy(state_dim, action_dim, hidden_dim).to(device)

# Optimisers
value_opt = optim.Adam(value_net.parameters(), lr=3e-4) # only updates value_net, value_target is updated via Polyak averaging
q_opt = optim.Adam(list(q1_net.parameters()) + list(q2_net.parameters()), lr=3e-4) # for both q1 and q2_net
policy_opt = optim.Adam(policy_net.parameters(), lr=3e-4) # for policy_net

In [ ]:
"""Train IQL on offline dataset"""

# Check if saved file exists
load = False
save_path = save_dir / "iql_model_w_c_1.pt"
if save_path.exists():
    load = True

if load:
    print("Loading IQL model from disk...")
    checkpoint = torch.load(save_path, map_location=device)
    value_net.load_state_dict(checkpoint["value_net"])
    value_target.load_state_dict(checkpoint["value_target"])
    q1_net.load_state_dict(checkpoint["q1_net"])
    q2_net.load_state_dict(checkpoint["q2_net"])
    policy_net.load_state_dict(checkpoint["policy_net"])

    value_opt.load_state_dict(checkpoint["value_opt"])
    q_opt.load_state_dict(checkpoint["q_opt"])
    policy_opt.load_state_dict(checkpoint["policy_opt"])

    s_mean = checkpoint["s_mean"]
    s_std = checkpoint["s_std"]
    a_mean = checkpoint["a_mean"]
    a_std = checkpoint["a_std"]

    start_epoch = checkpoint.get("epoch", 0)

    if additional_epochs <= 0 :
        print(f"Model loaded (epoch {start_epoch}). No additional training requested, skipping training.")
        do_train = False
    else:
        total_epochs = start_epoch + additional_epochs
        print(
            f"Model loaded (epoch {start_epoch}). "
            f"Continuing training for {additional_epochs} more epochs "
            f"(up to epoch {total_epochs})."
        )
        do_train = True
else:
    print("No existing checkpoint, training from scratch...")
    start_epoch = 0
    total_epochs = epochs
    do_train = True
    
if do_train:
    print("Training IQL model...")

    value_net.train()
    q1_net.train()
    q2_net.train()
    policy_net.train()

    for epoch in range(start_epoch, total_epochs):
        running_v_loss = 0.0
        running_q_loss = 0.0
        running_pi_loss = 0.0
        n_batches = 0

        for step, (s_b, a_b, r_b, s_next_b, done_b) in enumerate(tqdm(iql_loader, desc=f"Epoch {epoch+1}", miniters=100)):
            # Move batch to device and fix shapes
            s_b = s_b.to(device) # [batch_size, state_dim]
            a_b = a_b.to(device) # [batch_size, action_dim]
            r_b = r_b.to(device).squeeze(-1) # [batch_size]
            s_next_b = s_next_b.to(device) # [batch_size, state_dim]
            done_b = done_b.to(device).squeeze(-1) # [batch_size]

            # Value update (expectile regression)
            # No gradients because in the V update, the Q-values act as fixed targets (teacher)
            with torch.no_grad():
                # Compute Q-values under both critics
                q1_val = q1_net(s_b, a_b)
                q2_val = q2_net(s_b, a_b)
                # Take min to reduce overestimation bias
                q_min = torch.min(q1_val, q2_val)  # [batch_size]

            # Update V(s) using expectile regression: we compare Q(s,a) vs V(s)
            # If Q > V then the action looks better-than-typical for that state → pull V upward strongly (weight = τ)
            # If Q < V then the action looks worse-than-typical → pull V downward weakly (weight = 1−τ)
            # This makes V track the upper tail of Q and ignore low-return actions
            v = value_net(s_b)  # [batch_size]
            diff = q_min - v
            weight = torch.where(diff >= 0, tau_expectile, 1.0 - tau_expectile) # Choose weight depending on sign of diff
            value_loss = (weight * diff.pow(2)).mean() # Weighted squared loss → expectile regression

            value_opt.zero_grad()
            value_loss.backward()
            value_opt.step()

            # Q update (Bellman regression with V_target)
            # Now Q is the student: we freeze the target value network inside no_grad,
            # and update q1/q2 to match this fixed TD target
            with torch.no_grad():
                v_next = value_target(s_next_b)
                # done_b = 1 at episode end → no bootstrap term there (target = r only)
                target = r_b + gamma * (1.0 - done_b) * v_next

            q1 = q1_net(s_b, a_b)
            q2 = q2_net(s_b, a_b)
            # Sum TD errors from both critics per sample, then average over batch
            # Could also divide by 2 to get the mean per critic; it just rescales the loss
            q_loss = ((q1 - target).pow(2) + (q2 - target).pow(2)).mean()

            q_opt.zero_grad()
            q_loss.backward()
            q_opt.step()

            # Policy update (advantage-weighted behaviour cloning)
            with torch.no_grad():
                q1_pi = q1_net(s_b, a_b)
                q2_pi = q2_net(s_b, a_b)
                q_min_pi = torch.min(q1_pi, q2_pi)
                v = value_net(s_b)
                adv = q_min_pi - v # [batch_size], advantage
                weights = torch.exp(adv / beta) # beta is temperature parameter: larger beta → softer weighting
                weights = torch.clamp(weights, max=20.0) # Clamp so high-advantage samples don’t blow up gradients

            log_pi = policy_net.log_prob(s_b, a_b) # [batch_size], probability of the dataset action under the current policy π(a|s)
            policy_loss = -(weights * log_pi).mean() # If adv is positive → weight positive → policy pushed to imitate that action and vice-versa

            policy_opt.zero_grad()
            policy_loss.backward()
            policy_opt.step()

            # Target network update
            polyak_update(value_net, value_target, target_update_rate)

            # Track losses
            running_v_loss += value_loss.item()
            running_q_loss += q_loss.item()
            running_pi_loss += policy_loss.item()
            n_batches += 1

            if (step + 1) % log_every == 0:
                print(
                    f"Epoch {epoch+1} Step {step+1}: "
                    f"V_loss={running_v_loss / n_batches:.4f}, "
                    f"Q_loss={running_q_loss / n_batches:.4f}, "
                    f"Pi_loss={running_pi_loss / n_batches:.4f}"
                )

        print(
            f"\n\nEpoch {epoch+1} done."
            f"V_loss={running_v_loss / n_batches:.4f}, "
            f"Q_loss={running_q_loss / n_batches:.4f}, "
            f"Pi_loss={running_pi_loss / n_batches:.4f}"
        )
        print("---")

        # Save checkpoint after each epoch
        checkpoint = {
            "value_net": value_net.state_dict(),
            "value_target": value_target.state_dict(),
            "q1_net": q1_net.state_dict(),
            "q2_net": q2_net.state_dict(),
            "policy_net": policy_net.state_dict(),
            "value_opt": value_opt.state_dict(),
            "q_opt": q_opt.state_dict(),
            "policy_opt": policy_opt.state_dict(),
            "s_mean": s_mean,
            "s_std": s_std,
            "a_mean": a_mean,
            "a_std": a_std,
            "gamma": gamma,
            "tau_expectile": tau_expectile,
            "beta": beta,
            "epoch": epoch + 1,
        }

        # Always keep a "latest" checkpoint
        torch.save(checkpoint, save_path)

        # Also save a per-epoch checkpoint
        epoch_path = save_path.with_name(save_path.stem + f"_epoch{epoch+1}.pt")
        torch.save(checkpoint, epoch_path)
        print(f"Saved epoch {epoch+1} checkpoint to {epoch_path}")
    
    print(f"Saved final IQL model to {save_path}")

In [ ]:
"""Plot value map for different fixed other positions"""

# Setup geometry and metadata
nest_x = (float(nest_corner_1.X) + float(nest_corner_3.X)) / 2
nest_y = (float(nest_corner_1.Y) + float(nest_corner_3.Y)) / 2
patch1_x = (float(patch1_corner_1.X) + float(patch1_corner_3.X)) / 2
patch1_y = (float(patch1_corner_1.Y) + float(patch1_corner_3.Y)) / 2
patch2_x = (float(patch2_corner_1.X) + float(patch2_corner_3.X)) / 2
patch2_y = (float(patch2_corner_1.Y) + float(patch2_corner_3.Y)) / 2
patch3_x = (float(patch3_corner_1.X) + float(patch3_corner_3.X)) / 2
patch3_y = (float(patch3_corner_1.Y) + float(patch3_corner_3.Y)) / 2
corridor_r = (inner_radius + outer_radius) / 2

locations = {
    "Arena Center": (center_x, center_y),
    "Nest Area": (nest_x, nest_y),
    "Gate": (center_x - corridor_r, center_y),
    "Patch 1": (patch1_x, patch1_y),
    "Patch 2": (patch2_x, patch2_y),
    "Patch 3": (patch3_x, patch3_y)
}
n_locs = len(locations)

# Load image
arena_img = plt.imread("arena.png")
img_h, img_w = arena_img.shape[:2]
extent_img = [0, img_w, 0, img_h]

# Setup grid
nx, ny = 200, 200
xs = np.linspace(0, img_w, nx)
ys = np.linspace(0, img_h, ny)
xx, yy = np.meshgrid(xs, ys)

# Pre-calculate masks
valid_mask = get_validity_mask(xx, yy, metadata)

# Prepare figure
fig, axes = plt.subplots(
    n_configs, n_locs, 
    figsize=(4 * n_locs, 4 * n_configs), 
    squeeze=False,
    dpi=300
)

# Loop over configs (rows) and locations (columns)
unique_configs = list(config_counts.index)
for cfg_idx, cfg_tuple in enumerate(unique_configs):
    rates = [f"{rate:.4f}" for (_, rate) in cfg_tuple]
    cfg_name = f"({rates[0]}, {rates[1]}, {rates[2]})"

    # One-hot encoding for config
    c_grid = np.zeros((nx * ny, n_configs), dtype="float32")
    c_grid[:, cfg_idx] = 1.0
    c_grid_tensor = torch.from_numpy(c_grid).to(device)

    for loc_j, (loc_name, (other_x, other_y)) in enumerate(locations.items()):
        ax = axes[cfg_idx, loc_j]
        print(f"Computing masked values for config {cfg_name} with fixed other at {loc_name}...")

        # Construct state
        s_grid = np.zeros((nx * ny, 8), dtype="float32")
        s_grid[:, 0] = xx.ravel() # x_self
        s_grid[:, 1] = yy.ravel() # y_self
        s_grid[:, 4] = other_x - xx.ravel() # dx
        s_grid[:, 5] = other_y - yy.ravel() # dy
        
        # Normalize and query
        s_tensor = torch.from_numpy(s_grid).to(device)
        s_norm = (s_tensor - s_mean.to(device)) / s_std.to(device)

        s_full = torch.cat([s_norm, c_grid_tensor], dim=1)

        value_net.eval()
        with torch.no_grad():
            vals = value_net(s_full).cpu().numpy()
        
        # Reshape
        v_vals = vals.reshape(ny, nx)

        # Apply mask
        v_vals_masked = np.where(valid_mask, v_vals, np.nan)

        # Plot background
        if arena_img is not None:
            ax.imshow(arena_img, origin="lower", alpha=1.0, extent=extent_img)

        # Plot masked values
        im = ax.imshow(
            v_vals_masked,
            origin="lower",
            extent=[0, img_w, 0, img_h], # Use full image bounds
            cmap="viridis",
            alpha=0.6
        )
        
        # Add colorbar to every plot (needed since value scales might differ)
        # We make it slightly smaller using 'fraction' to keep the grid tidy
        fig.colorbar(im, ax=ax, label="Value V(s)", fraction=0.046, pad=0.04)

        # Plot fixed other mouse
        ax.plot(other_x, other_y, 'b*', markersize=25, markerfacecolor='none', markeredgecolor='white', label=f"Other ({loc_name})")

        # Labeling Logic:
        # Column Labels (Top Row only)
        if cfg_idx == 0:
            ax.set_title(loc_name, fontsize=14, fontweight='bold')
        else:
            ax.set_title("") 

        # Row Labels (Left Column only)
        if loc_j == 0:
            # Rotated 90 degrees
            ax.set_ylabel(f"Config\n{cfg_name}", fontsize=12, fontweight='bold', rotation=90, labelpad=20)
            ax.set_yticks([]) 
        else:
            ax.set_ylabel("")
            ax.set_yticks([])

        # Remove x ticks for all plots
        ax.set_xticks([])
        
        ax.set_xlim(0, img_w)
        ax.set_ylim(0, img_h)
        ax.set_aspect('equal')

# Add a single legend for the whole figure
legend_elements = [Line2D([0], [0], marker='*', color='w', label='Other Mouse',
                          markerfacecolor='none', markeredgecolor='b', markersize=15)]
fig.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(0.0, 1.0))

plt.tight_layout()
plt.show()

In [ ]:
"""Plot navigation policy for different fixed other positions"""

# Define locations
nest_x = (float(nest_corner_1.X) + float(nest_corner_3.X)) / 2
nest_y = (float(nest_corner_1.Y) + float(nest_corner_3.Y)) / 2
patch1_x = (float(patch1_corner_1.X) + float(patch1_corner_3.X)) / 2
patch1_y = (float(patch1_corner_1.Y) + float(patch1_corner_3.Y)) / 2
patch2_x = (float(patch2_corner_1.X) + float(patch2_corner_3.X)) / 2
patch2_y = (float(patch2_corner_1.Y) + float(patch2_corner_3.Y)) / 2
patch3_x = (float(patch3_corner_1.X) + float(patch3_corner_3.X)) / 2
patch3_y = (float(patch3_corner_1.Y) + float(patch3_corner_3.Y)) / 2
corridor_r = (inner_radius + outer_radius) / 2

locations = {
    "Arena Center": (center_x, center_y),
    "Nest Area": (nest_x, nest_y),
    "Gate": (center_x - corridor_r, center_y),
    "Patch 1": (patch1_x, patch1_y),
    "Patch 2": (patch2_x, patch2_y),
    "Patch 3": (patch3_x, patch3_y)
}
n_locs = len(locations)

# Load image and determine global bounds
arena_img = plt.imread("arena.png")
img_h, img_w = arena_img.shape[:2]
extent_img = [0, img_w, 0, img_h] # Image uses its own full coords

# Setup grid (Full Image)
nx, ny = 30, 30
xs = np.linspace(0, img_w, nx)
ys = np.linspace(0, img_h, ny)
xx, yy = np.meshgrid(xs, ys)

# Prepare figure
fig, axes = plt.subplots(
    n_configs, n_locs, 
    figsize=(4 * n_locs, 4 * n_configs), 
    squeeze=False,
    dpi=300
)

# Loop over configs (rows) and locations (columns)
unique_configs = list(config_counts.index)
for cfg_idx, cfg_tuple in enumerate(unique_configs):
    rates = [f"{rate:.4f}" for (_, rate) in cfg_tuple]
    # Format: (Rate1, Rate2, Rate3)
    cfg_name = f"({rates[0]}, {rates[1]}, {rates[2]})"
    
    # One-hot encoding for config
    c_grid = np.zeros((nx * ny, n_configs), dtype="float32")
    c_grid[:, cfg_idx] = 1.0
    c_grid_tensor = torch.from_numpy(c_grid).to(device)

    for loc_j, (loc_name, (other_x, other_y)) in enumerate(locations.items()):
        ax = axes[cfg_idx, loc_j]
        
        print(f"Computing policy for config {cfg_name} with fixed other at {loc_name}...")

        # Construct state
        s_grid = np.zeros((nx * ny, 8), dtype="float32")
        s_grid[:, 0] = xx.ravel() # x_self
        s_grid[:, 1] = yy.ravel() # y_self
        s_grid[:, 4] = other_x - xx.ravel() # dx
        s_grid[:, 5] = other_y - yy.ravel() # dy
        
        # Normalize
        s_tensor = torch.from_numpy(s_grid).to(device)
        s_norm = (s_tensor - s_mean.to(device)) / s_std.to(device)

        s_full = torch.cat([s_norm, c_grid_tensor], dim=1)

        # Query IQL policy
        policy_net.eval()
        with torch.no_grad():
            mu, _ = policy_net(s_full)
            mu = mu.cpu().numpy()

        # Un-normalize actions
        a_std_np = a_std.detach().cpu().numpy()
        a_mean_np = a_mean.detach().cpu().numpy()
        actions = (mu * a_std_np) + a_mean_np

        u = actions[:, 0].reshape(ny, nx)
        v = actions[:, 1].reshape(ny, nx)

        # Apply masking
        valid_start = get_validity_mask(xx, yy, metadata)
        u = np.where(valid_start, u, np.nan)
        v = np.where(valid_start, v, np.nan)

        # Plot background (use full image extent)
        if arena_img is not None:
            ax.imshow(arena_img, origin="lower", alpha=0.5, extent=extent_img)

        # Plot self movement (arrows)
        ax.quiver(xx, yy, u, v, color='red', scale=None, scale_units='inches')

        # Plot fixed other mouse (star)
        ax.plot(other_x, other_y, 'b*', markersize=25, markeredgecolor='white', label=f"Other ({loc_name})")

        # Labeling Logic:
        # Column Labels (Top Row only)
        if cfg_idx == 0:
            ax.set_title(loc_name, fontsize=14, fontweight='bold')
        else:
            ax.set_title("") 

        # Row Labels (Left Column only)
        if loc_j == 0:
            # Rotated 90 degrees
            ax.set_ylabel(f"Config\n{cfg_name}", fontsize=12, fontweight='bold', rotation=90, labelpad=20)
            # Remove standard y ticks to clean up
            ax.set_yticks([]) 
        else:
            ax.set_ylabel("")
            ax.set_yticks([])

        # Remove x ticks for all plots to clean up
        ax.set_xticks([])
        
        # Lock view to full arena dimensions
        ax.set_xlim(0, img_w)
        ax.set_ylim(0, img_h)
        
        ax.set_aspect('equal')

# Add a single legend for the whole figure
legend_elements = [Line2D([0], [0], marker='*', color='w', label='Other Mouse',
                          markerfacecolor='b', markersize=15)]
fig.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(0.0, 1.0))

plt.tight_layout()
plt.show()